# Predicting Heart Disease - S6E2
## v8: Pseudo Labeling

v4の特徴量 + CatBoostで予測 → 高確信サンプルをtrainに追加 → 再学習

In [1]:
import numpy as np
import pandas as pd
import gc
import warnings
from itertools import combinations

from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata

warnings.filterwarnings('ignore')

In [2]:
class CFG:
    seed = 42
    n_splits = 5
    target_col = 'Heart Disease'
    use_original = True
    n_top_pairs = 8
    # Pseudo Labeling 閾値
    pl_high = 0.95  # これ以上 → y=1
    pl_low = 0.05   # これ以下 → y=0

class Paths:
    p = '/Users/shirokoshikentaro/Desktop/Predicting Heart Disease/Data'
    train = p + '/train.csv'
    test = p + '/test.csv'
    sample = p + '/sample_submission.csv'
    original = p + '/dataset_heart.csv'
    public_sub = p + '/submission.csv'

## 1. Data Loading + Feature Engineering（v4と同一）

In [3]:
train = pd.read_csv(Paths.train)
test = pd.read_csv(Paths.test)
sample_submission = pd.read_csv(Paths.sample)

if CFG.use_original:
    original = pd.read_csv(Paths.original)
    rename_map = {
        'age': 'Age', 'sex ': 'Sex',
        'chest pain type': 'Chest pain type',
        'resting blood pressure': 'BP',
        'serum cholestoral': 'Cholesterol',
        'fasting blood sugar': 'FBS over 120',
        'resting electrocardiographic results': 'EKG results',
        'max heart rate': 'Max HR',
        'exercise induced angina': 'Exercise angina',
        'oldpeak': 'ST depression',
        'ST segment': 'Slope of ST',
        'major vessels': 'Number of vessels fluro',
        'thal': 'Thallium',
        'heart disease': 'Heart Disease',
    }
    original = original.rename(columns=rename_map)
    original['Heart Disease'] = original['Heart Disease'].map({1: 0, 2: 1})
    if 'id' not in original.columns:
        max_id = train['id'].max()
        original['id'] = range(max_id + 1, max_id + 1 + len(original))
    original = original[train.columns]
    train = pd.concat([train, original], axis=0, ignore_index=True)
    cols_for_dedup = [c for c in train.columns if c != 'id']
    train = train.drop_duplicates(subset=cols_for_dedup, keep='first').reset_index(drop=True)

TARGET = CFG.target_col
if train[TARGET].dtype == 'object' or train[TARGET].dtype.name == 'category':
    train[TARGET] = train[TARGET].map({'Absence': 0, 'Presence': 1})

y = train[TARGET].values.astype(np.uint8)
test_ids = test['id']

X_tr_raw = train.drop(columns=[TARGET, 'id'])
X_te_raw = test.drop(columns=['id'])

cat_cols = [
    'Sex', 'Chest pain type', 'FBS over 120', 'EKG results',
    'Exercise angina', 'Slope of ST', 'Number of vessels fluro', 'Thallium'
]
num_cols = ['Age', 'BP', 'Cholesterol', 'Max HR', 'ST depression']
all_base_cols = cat_cols + num_cols

print(f'Train: {train.shape}, Test: {test.shape}')

Train: (630270, 15), Test: (270000, 14)


In [4]:
skf_te = StratifiedKFold(n_splits=5, shuffle=True, random_state=CFG.seed)

# --- Frequency Encoding ---
all_df = pd.concat([X_tr_raw, X_te_raw])
tr_freq = pd.DataFrame(index=X_tr_raw.index)
te_freq = pd.DataFrame(index=X_te_raw.index)
for c in all_base_cols:
    freq = all_df[c].value_counts(normalize=True)
    tr_freq[c + '_freq'] = X_tr_raw[c].map(freq)
    te_freq[c + '_freq'] = X_te_raw[c].map(freq)

# --- Target Encoding (OOF) ---
tr_te = pd.DataFrame(index=X_tr_raw.index)
te_te = pd.DataFrame(index=X_te_raw.index)
for c in all_base_cols:
    tr_te[c + '_te'] = 0.0
    for tr_i, val_i in skf_te.split(X_tr_raw, y):
        means = train.iloc[tr_i].groupby(c)[TARGET].mean()
        tr_te.iloc[val_i, tr_te.columns.get_loc(c + '_te')] = X_tr_raw.iloc[val_i][c].map(means)
    te_te[c + '_te'] = X_te_raw[c].map(train.groupby(c)[TARGET].mean())

# --- Feature Growth ---
base = tr_te.fillna(0)
corr_scores = {}
for a, b in combinations(base.columns, 2):
    corr_scores[(a, b)] = abs(np.corrcoef(base[a], base[b])[0, 1])
top_pairs = sorted(corr_scores, key=corr_scores.get, reverse=True)[:CFG.n_top_pairs]
for a, b in top_pairs:
    tr_te[f'{a}_x_{b}'] = tr_te[a] * tr_te[b]
    te_te[f'{a}_x_{b}'] = te_te[a] * te_te[b]

# --- 手動交互作用 ---
def add_manual_features(df):
    out = pd.DataFrame(index=df.index)
    for col in df.select_dtypes(include=['int8', 'int16']).columns:
        df[col] = df[col].astype('int32')
    for col in df.select_dtypes(include=['float16']).columns:
        df[col] = df[col].astype('float32')
    out['Age_x_MaxHR'] = df['Age'] * df['Max HR']
    out['Age_div_MaxHR'] = df['Age'] / (df['Max HR'] + 1e-5)
    out['STdep_x_MaxHR'] = df['ST depression'] * df['Max HR']
    out['BP_x_Chol'] = df['BP'] * df['Cholesterol']
    out['Age_x_STdep'] = df['Age'] * df['ST depression']
    out['Age_x_Vessels'] = df['Age'] * df['Number of vessels fluro']
    out['BP_x_MaxHR'] = df['BP'] * df['Max HR']
    out['Chol_x_Age'] = df['Cholesterol'] * df['Age']
    out['STdep_x_Vessels'] = df['ST depression'] * df['Number of vessels fluro']
    out['Sex_x_MaxHR'] = df['Sex'] * df['Max HR']
    out['Sex_x_Chol'] = df['Sex'] * df['Cholesterol']
    out['ChestPain_x_STdep'] = df['Chest pain type'] * df['ST depression']
    out['ChestPain_x_MaxHR'] = df['Chest pain type'] * df['Max HR']
    out['ExAngina_x_STdep'] = df['Exercise angina'] * df['ST depression']
    out['ExAngina_x_MaxHR'] = df['Exercise angina'] * df['Max HR']
    out['Thallium_x_Vessels'] = df['Thallium'] * df['Number of vessels fluro']
    out['SlopeSTx_STdep'] = df['Slope of ST'] * df['ST depression']
    out['MaxHR_reserve'] = (220 - df['Age']) - df['Max HR']
    out['MaxHR_ratio'] = df['Max HR'] / (220 - df['Age'] + 1e-5)
    return out

tr_manual = add_manual_features(X_tr_raw)
te_manual = add_manual_features(X_te_raw)

# --- 結合 ---
X_train = pd.concat([X_tr_raw, tr_freq, tr_te, tr_manual], axis=1).fillna(0)
X_test = pd.concat([X_te_raw, te_freq, te_te, te_manual], axis=1).fillna(0)

print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')

X_train: (630270, 66), X_test: (270000, 66)


## 2. Step 1: 初回学習（Pseudo Label生成用）

In [5]:
cat_params = {
    'iterations': 3000,
    'learning_rate': 0.03,
    'depth': 4,
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'auto_class_weights': 'Balanced',
    'subsample': 0.65,
    'l2_leaf_reg': 12,
    'random_strength': 1.2,
    'bootstrap_type': 'Bernoulli',
    'task_type': 'CPU',
    'verbose': False,
}

skf = StratifiedKFold(n_splits=CFG.n_splits, shuffle=True, random_state=CFG.seed)

# 初回予測（Pseudo Label用）
test_pred_step1 = np.zeros(len(X_test))
oof_step1 = np.zeros(len(X_train))

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y)):
    cb = CatBoostClassifier(**cat_params, random_seed=CFG.seed + fold)
    cb.fit(X_train.iloc[tr_idx], y[tr_idx])
    oof_step1[va_idx] = cb.predict_proba(X_train.iloc[va_idx])[:, 1]
    test_pred_step1 += cb.predict_proba(X_test)[:, 1] / CFG.n_splits

step1_auc = roc_auc_score(y, oof_step1)
print(f'Step 1 OOF AUC: {step1_auc:.6f}')

Step 1 OOF AUC: 0.955402


## 3. Pseudo Label 生成

In [6]:
# 高確信サンプルを抽出
high_conf_pos = test_pred_step1 > CFG.pl_high  # ほぼ確実に心臓病
high_conf_neg = test_pred_step1 < CFG.pl_low   # ほぼ確実に健康
high_conf = high_conf_pos | high_conf_neg

pseudo_X = X_test[high_conf].copy()
pseudo_y = (test_pred_step1[high_conf] > 0.5).astype(np.uint8)

print(f'=== Pseudo Labeling ===')
print(f'  閾値: > {CFG.pl_high} → y=1, < {CFG.pl_low} → y=0')
print(f'  Positive (y=1): {high_conf_pos.sum():,}')
print(f'  Negative (y=0): {high_conf_neg.sum():,}')
print(f'  合計: {high_conf.sum():,} / {len(X_test):,} ({high_conf.mean()*100:.1f}%)')
print(f'\n  Train: {len(X_train):,} → {len(X_train) + len(pseudo_X):,} (+{len(pseudo_X):,})')

=== Pseudo Labeling ===
  閾値: > 0.95 → y=1, < 0.05 → y=0
  Positive (y=1): 62,785
  Negative (y=0): 71,499
  合計: 134,284 / 270,000 (49.7%)

  Train: 630,270 → 764,554 (+134,284)


## 4. Step 2: Pseudo Label込みで再学習

In [7]:
# trainとpseudo labelを結合
X_train_pl = pd.concat([X_train, pseudo_X], axis=0, ignore_index=True)
y_pl = np.concatenate([y, pseudo_y])

print(f'X_train_pl: {X_train_pl.shape}')
print(f'y_pl distribution: 0={np.mean(y_pl==0):.3f}, 1={np.mean(y_pl==1):.3f}')

# 再学習（元のtrainに対するOOFも計算）
oof_step2 = np.zeros(len(X_train))  # 元のtrainのみでOOF計算
test_pred_step2 = np.zeros(len(X_test))

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y)):
    # trainのfold分割 + pseudo label全体を学習データに追加
    pseudo_indices = np.arange(len(X_train), len(X_train_pl))
    tr_idx_pl = np.concatenate([tr_idx, pseudo_indices])

    cb = CatBoostClassifier(**cat_params, random_seed=CFG.seed + fold)
    cb.fit(X_train_pl.iloc[tr_idx_pl], y_pl[tr_idx_pl])

    # validationは元のtrainのみ（公正な評価）
    oof_step2[va_idx] = cb.predict_proba(X_train.iloc[va_idx])[:, 1]
    test_pred_step2 += cb.predict_proba(X_test)[:, 1] / CFG.n_splits

step2_auc = roc_auc_score(y, oof_step2)
print(f'\nStep 1 OOF AUC: {step1_auc:.6f}')
print(f'Step 2 OOF AUC: {step2_auc:.6f}')
print(f'Improvement:     {step2_auc - step1_auc:+.6f}')

X_train_pl: (764554, 66)
y_pl distribution: 0=0.548, 1=0.452

Step 1 OOF AUC: 0.955402
Step 2 OOF AUC: 0.954563
Improvement:     -0.000839


## 5. 閾値を変えて複数パターン試す

In [8]:
# 閾値を変えた場合の効果も確認
thresholds = [
    (0.90, 0.10, 'loose'),
    (0.95, 0.05, 'default'),
    (0.98, 0.02, 'strict'),
    (0.99, 0.01, 'very_strict'),
]

results = {}

for pl_high, pl_low, name in thresholds:
    mask = (test_pred_step1 > pl_high) | (test_pred_step1 < pl_low)
    px = X_test[mask].copy()
    py = (test_pred_step1[mask] > 0.5).astype(np.uint8)

    X_pl = pd.concat([X_train, px], axis=0, ignore_index=True)
    y_combined = np.concatenate([y, py])

    oof_tmp = np.zeros(len(X_train))
    test_tmp = np.zeros(len(X_test))

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y)):
        pseudo_idx = np.arange(len(X_train), len(X_pl))
        tr_idx_all = np.concatenate([tr_idx, pseudo_idx])

        cb = CatBoostClassifier(**cat_params, random_seed=CFG.seed + fold)
        cb.fit(X_pl.iloc[tr_idx_all], y_combined[tr_idx_all])

        oof_tmp[va_idx] = cb.predict_proba(X_train.iloc[va_idx])[:, 1]
        test_tmp += cb.predict_proba(X_test)[:, 1] / CFG.n_splits

    auc = roc_auc_score(y, oof_tmp)
    results[name] = {'auc': auc, 'test_pred': test_tmp, 'n_pseudo': mask.sum(),
                     'pl_high': pl_high, 'pl_low': pl_low}
    print(f'  {name:12s} (>{pl_high}, <{pl_low}): PL={mask.sum():5d}, OOF AUC={auc:.6f}, diff={auc-step1_auc:+.6f}')

# ベスト閾値を選択
best_name = max(results, key=lambda k: results[k]['auc'])
best = results[best_name]
print(f'\n★ Best: {best_name} (OOF AUC={best["auc"]:.6f})')

  loose        (>0.9, <0.1): PL=172626, OOF AUC=0.954004, diff=-0.001398


KeyboardInterrupt: 

## 6. Submissions

In [ ]:
# ベスト閾値のPseudo Label結果
sub_pl = pd.DataFrame({'id': test_ids, TARGET: best['test_pred'].astype(float)})
sub_pl.to_csv('submission_v8_pseudo_label.csv', index=False)
print(f'✅ submission_v8_pseudo_label.csv ({best_name})')

# Public Notebookとのブレンド（ファイルがある場合）
import os

def rank_avg_2(p1, p2, w1):
    r1 = rankdata(p1) / len(p1)
    r2 = rankdata(p2) / len(p2)
    return w1 * r1 + (1 - w1) * r2

if os.path.exists(Paths.public_sub):
    public_sub = pd.read_csv(Paths.public_sub)
    public_pred = public_sub[TARGET].values

    print('\n--- Public Blends ---')
    for w in [0.1, 0.2, 0.3]:
        blended = rank_avg_2(best['test_pred'], public_pred, w)
        sub = pd.DataFrame({'id': test_ids, TARGET: blended.astype(float)})
        fname = f'submission_v8_blend_pl{int(w*100)}_pub{int((1-w)*100)}.csv'
        sub.to_csv(fname, index=False)
        print(f'  ✅ {fname}')
else:
    print('\n⚠️ Public submission not found. Skipping blend.')

In [ ]:
print('=' * 60)
print('FINAL SUMMARY (v8 Pseudo Labeling)')
print('=' * 60)
print(f'  Step 1 OOF AUC:      {step1_auc:.6f}')
print(f'  Best PL OOF AUC:     {best["auc"]:.6f}')
print(f'  Improvement:         {best["auc"] - step1_auc:+.6f}')
print(f'  Best threshold:      >{best["pl_high"]}, <{best["pl_low"]} ({best_name})')
print(f'  Pseudo samples:      {best["n_pseudo"]}')
print('=' * 60)
print('\n→ submission_v8_pseudo_label.csv を提出してください')
print('  前回ベスト: 0.95387（ブレンド）/ 0.95376（CatBoost単体）')